# Programma 2 — Estrazione di informazioni

Estrazione di informazioni dal **Corpus 1** (`Corpora_1.txt`): prosa letteraria, estratto da *Pride and Prejudice* di Jane Austen.

In [1]:
# Import delle librerie e download delle risorse NLTK

import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.tag import pos_tag
from nltk.corpus import stopwords
from nltk import FreqDist, bigrams, ngrams, ne_chunk
import math

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('stopwords')
nltk.download('maxent_ne_chunker')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     C:\Users\user\AppData\Roaming\

True

## Lettura del corpus

In [2]:
# Lettura del corpus da analizzare
# Per analizzare l'altro corpus è sufficiente cambiare il nome del file e rieseguire

nome_file = "Corpora_1.txt"

with open(nome_file, "r", encoding="utf-8") as f:
    testo = f.read()

# Verifica: stampa dei primi 200 caratteri
print(f"=== Inizio di {nome_file} ===")
print(testo[:200])

=== Inizio di Corpora_1.txt ===
It is a truth universally acknowledged, that a single man in possession of a good fortune must be in want of a wife. However little known the feelings or views of such a man may be on his first enteri


## Preprocessing

In [3]:
# Suddivisione del testo in frasi
frasi = sent_tokenize(testo)

# Tokenizzazione: lista piatta di tutti i token del corpus
tokens_con_punteggiatura = [token for frase in frasi for token in word_tokenize(frase)]

# PoS tagging dell'intero corpus
pos_con_punteggiatura = pos_tag(tokens_con_punteggiatura)

# Rimozione della punteggiatura: si tengono solo i token che contengono
# almeno un carattere alfanumerico. In questo modo vengono esclusi sia i segni
# di punteggiatura standard sia quelli tipografici.
tokens_pos = [(token, tag) for token, tag in pos_con_punteggiatura if any(c.isalnum() for c in token)]

# Token nella forma originale (con maiuscole), usati per le Named Entity
tokens = [token for token, tag in tokens_pos]

# Token normalizzati in minuscolo, usati per i conteggi lessicali
tokens_lower = [token.lower() for token in tokens]

print(f"Numero di frasi: {len(frasi)}")
print(f"Numero di token (senza punteggiatura): {len(tokens)}")

Numero di frasi: 626
Numero di token (senza punteggiatura): 17523


## 1. Top-50 sostantivi, verbi e aggettivi più frequenti

In [4]:
# Funzione che estrae i top-N token appartenenti a una categoria grammaticale,
# ordinati per frequenza decrescente. I tag PoS da considerare sono passati come lista
# I token vengono normalizzati in minuscolo per non distinguere le maiuscole

def top_per_categoria(tokens_pos, tag_validi, n=50):
    parole = [token.lower() for token, tag in tokens_pos if tag in tag_validi]
    return FreqDist(parole).most_common(n)

# Per i sostantivi si considerano tutti i tag nominali (nomi comuni e propri)
tag_sostantivi = ["NN", "NNS", "NNP", "NNPS"]
tag_verbi = ["VB", "VBD", "VBG", "VBN", "VBP", "VBZ"]
tag_aggettivi = ["JJ", "JJR", "JJS"]

top50_sostantivi = top_per_categoria(tokens_pos, tag_sostantivi)
top50_verbi = top_per_categoria(tokens_pos, tag_verbi)
top50_aggettivi = top_per_categoria(tokens_pos, tag_aggettivi)

print("=== Top-50 Sostantivi ===")
for parola, freq in top50_sostantivi:
    print(f"{parola}: {freq}")

print("\n=== Top-50 Verbi ===")
for parola, freq in top50_verbi:
    print(f"{parola}: {freq}")

print("\n=== Top-50 Aggettivi ===")
for parola, freq in top50_aggettivi:
    print(f"{parola}: {freq}")

=== Top-50 Sostantivi ===
mr.: 144
bingley: 99
bennet: 86
darcy: 74
elizabeth: 69
miss: 69
mrs.: 58
jane: 44
man: 27
girls: 23
friend: 22
room: 22
time: 22
daughters: 21
s: 21
nothing: 21
lady: 20
netherfield: 20
sister: 20
lucas: 19
mother: 19
sir: 18
ladies: 18
pleasure: 18
day: 17
lydia: 17
ball: 17
hurst: 17
attention: 17
woman: 16
way: 16
evening: 16
sisters: 16
oh: 15
manner: 15
morning: 15
family: 15
catherine: 15
subject: 15
house: 14
something: 14
father: 14
meryton: 14
answer: 13
dear: 13
everybody: 13
eyes: 13
country: 13
person: 12
people: 12

=== Top-50 Verbi ===
was: 208
is: 189
be: 162
had: 158
have: 94
are: 82
said: 81
do: 67
were: 66
am: 57
been: 49
know: 39
think: 33
did: 33
has: 32
s: 29
cried: 25
say: 25
being: 25
made: 23
see: 22
does: 22
make: 18
go: 16
replied: 14
wish: 13
give: 12
assure: 12
believe: 12
let: 11
hope: 11
having: 11
received: 11
tell: 10
dare: 10
come: 10
continued: 10
dance: 10
mean: 10
thought: 10
keep: 10
write: 10
heard: 9
take: 9
consider: 9


## 2. Top-10 n-grammi più frequenti

In [5]:
# Funzione che estrae i top-10 n-grammi più frequenti per un dato n
# Si lavora sui token normalizzati in minuscolo (tokens_lower)

def top_ngrammi(tokens, n, top=10):
    n_grammi = list(ngrams(tokens, n))
    return FreqDist(n_grammi).most_common(top)

# Calcolo per n = 1, 2, 3
for n in [1, 2, 3]:
    print(f"=== Top-10 {n}-grammi ===")
    for ngramma, freq in top_ngrammi(tokens_lower, n):
        print(f"{ngramma}: {freq}")
    print()

=== Top-10 1-grammi ===
('the',): 595
('to',): 561
('and',): 495
('of',): 479
('i',): 356
('a',): 352
('in',): 272
('you',): 261
('her',): 250
('not',): 241

=== Top-10 2-grammi ===
('in', 'the'): 65
('to', 'be'): 64
('of', 'the'): 58
('i', 'am'): 54
('it', 'is'): 46
('mr.', 'bingley'): 44
('mr.', 'darcy'): 42
('mrs.', 'bennet'): 35
('miss', 'bingley'): 35
('he', 'had'): 34

=== Top-10 3-grammi ===
('i', 'do', 'not'): 18
('i', 'am', 'sure'): 13
('it', 'is', 'a'): 9
('it', 'would', 'be'): 9
('i', 'dare', 'say'): 9
('i', 'can', 'not'): 7
('as', 'soon', 'as'): 6
('said', 'mr.', 'bennet'): 6
('it', 'is', 'not'): 6
('to', 'be', 'a'): 6



## 3. Top-30 n-grammi di PoS più frequenti

In [6]:
# Funzione che estrae i top-30 n-grammi di PoS più frequenti per un dato n
# Si considerano solo le sequenze di categorie grammaticali (i tag), non i token

def top_ngrammi_pos(tokens_pos, n, top=30):
    soli_tag = [tag for token, tag in tokens_pos]
    n_grammi = list(ngrams(soli_tag, n))
    return FreqDist(n_grammi).most_common(top)

# Calcolo per n = 1, 2, 3, 4, 5
for n in [1, 2, 3, 4, 5]:
    print(f"=== Top-30 {n}-grammi di PoS ===")
    for ngramma, freq in top_ngrammi_pos(tokens_pos, n):
        print(f"{ngramma}: {freq}")
    print()

=== Top-30 1-grammi di PoS ===
('NN',): 2151
('IN',): 2121
('PRP',): 1673
('DT',): 1314
('RB',): 1238
('JJ',): 1124
('NNP',): 1090
('VBD',): 1012
('VB',): 874
('CC',): 697
('PRP$',): 649
('TO',): 561
('VBP',): 477
('VBN',): 433
('NNS',): 432
('MD',): 374
('VBZ',): 309
('VBG',): 302
('CD',): 115
('WP',): 103
('WRB',): 100
('WDT',): 87
('JJS',): 58
('JJR',): 50
('EX',): 39
('RBR',): 36
('RP',): 33
('PDT',): 31
('RBS',): 21
('UH',): 10

=== Top-30 2-grammi di PoS ===
('DT', 'NN'): 749
('NN', 'IN'): 694
('IN', 'DT'): 543
('IN', 'PRP'): 397
('PRP$', 'NN'): 381
('JJ', 'NN'): 379
('PRP', 'VBD'): 379
('TO', 'VB'): 361
('NNP', 'NNP'): 351
('PRP', 'VBP'): 347
('NN', 'CC'): 285
('IN', 'PRP$'): 283
('DT', 'JJ'): 261
('IN', 'NN'): 249
('PRP', 'MD'): 240
('MD', 'VB'): 236
('RB', 'JJ'): 215
('NN', 'PRP'): 203
('VBD', 'RB'): 198
('NNP', 'VBD'): 197
('RB', 'VB'): 162
('IN', 'NNP'): 152
('NN', 'NNP'): 152
('VBN', 'IN'): 151
('VB', 'PRP'): 151
('JJ', 'IN'): 150
('RB', 'IN'): 142
('NN', 'VBD'): 138
('VB',

## 4. Top-10 bigrammi Aggettivo + Sostantivo

In [7]:
# Estrazione dei bigrammi Aggettivo + Sostantivo dal corpus
# Si scorrono i bigrammi di token consecutivi e si tengono solo quelli in cui
# il primo token è un aggettivo (JJ, JJR, JJS) e il secondo un sostantivo (NN, NNS, NNP, NNPS)

tag_agg = ["JJ", "JJR", "JJS"]
tag_sost = ["NN", "NNS", "NNP", "NNPS"]

bigrammi_pos = list(bigrams(tokens_pos))

bigrammi_agg_sost = []
for (token1, tag1), (token2, tag2) in bigrammi_pos:
    if tag1 in tag_agg and tag2 in tag_sost:
        # il bigramma viene salvato come coppia di parole in minuscolo
        bigrammi_agg_sost.append((token1.lower(), token2.lower()))

# Frequenza dei bigrammi Agg+Sost
freq_bigrammi = FreqDist(bigrammi_agg_sost)

# 4.1 - Top-10 per frequenza decrescente
print("=== Top-10 bigrammi Agg+Sost per frequenza ===")
for bigramma, freq in freq_bigrammi.most_common(10):
    print(f"{bigramma}: {freq}")

=== Top-10 bigrammi Agg+Sost per frequenza ===
('young', 'man'): 6
('dear', 'mr.'): 4
('young', 'lady'): 4
('miss', 'bennet'): 4
('young', 'ladies'): 4
('large', 'fortune'): 3
('own', 'children'): 3
('great', 'deal'): 3
('same', 'time'): 3
('young', 'woman'): 3


In [8]:
# Per calcolare le probabilità servono le frequenze dei singoli token (in minuscolo)
# e il numero totale di token del corpus
freq_token = FreqDist(tokens_lower)
N = len(tokens_lower)

# 4.2 - Probabilità condizionata: P(sostantivo | aggettivo) = freq(agg, sost) / freq(agg)
prob_condizionata = {}
for bigramma, freq in freq_bigrammi.items():
    aggettivo = bigramma[0]
    prob_condizionata[bigramma] = freq / freq_token[aggettivo]

# 4.3 - Probabilità congiunta: P(aggettivo, sostantivo) = freq(agg, sost) / N
prob_congiunta = {}
for bigramma, freq in freq_bigrammi.items():
    prob_congiunta[bigramma] = freq / N

# Ordinamento e stampa dei top-10 per probabilità condizionata
print("=== Top-10 bigrammi Agg+Sost per probabilità condizionata ===")
for bigramma, prob in sorted(prob_condizionata.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"{bigramma}: {prob:.4f}")

# Ordinamento e stampa dei top-10 per probabilità congiunta
print("\n=== Top-10 bigrammi Agg+Sost per probabilità congiunta ===")
for bigramma, prob in sorted(prob_congiunta.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"{bigramma}: {prob:.6f}")

=== Top-10 bigrammi Agg+Sost per probabilità condizionata ===
('rightful', 'property'): 1.0000
('grown-up', 'daughters'): 1.0000
('hearty', 'consent'): 1.0000
('sisters.', 'mr'): 1.0000
('least.', 'ah'): 1.0000
('quick', 'parts'): 1.0000
('sarcastic', 'humour'): 1.0000
('three-and-twenty', 'years'): 1.0000
('uncertain', 'temper'): 1.0000
('heaven', 's'): 1.0000

=== Top-10 bigrammi Agg+Sost per probabilità congiunta ===
('young', 'man'): 0.000342
('dear', 'mr.'): 0.000228
('young', 'lady'): 0.000228
('miss', 'bennet'): 0.000228
('young', 'ladies'): 0.000228
('large', 'fortune'): 0.000171
('own', 'children'): 0.000171
('great', 'deal'): 0.000171
('same', 'time'): 0.000171
('young', 'woman'): 0.000171


In [9]:
# 4.4 - MI (Mutual Information) e LMI (Local Mutual Information)
# MI(A,B)  = log2( freq(A,B) * N / (freq(A) * freq(B)) )
# LMI(A,B) = freq(A,B) * MI(A,B)

mi = {}
lmi = {}
for bigramma, freq in freq_bigrammi.items():
    aggettivo, sostantivo = bigramma
    # MI: rapporto tra co-occorrenza osservata e attesa, in scala logaritmica (base 2)
    valore_mi = math.log2((freq * N) / (freq_token[aggettivo] * freq_token[sostantivo]))
    mi[bigramma] = valore_mi
    # LMI: MI pesata per la frequenza del bigramma
    lmi[bigramma] = freq * valore_mi

# Top-10 per MI
print("=== Top-10 bigrammi Agg+Sost per MI ===")
top_mi = sorted(mi.items(), key=lambda x: x[1], reverse=True)[:10]
for bigramma, valore in top_mi:
    print(f"{bigramma}: {valore:.4f}")

# Top-10 per LMI
print("\n=== Top-10 bigrammi Agg+Sost per LMI ===")
top_lmi = sorted(lmi.items(), key=lambda x: x[1], reverse=True)[:10]
for bigramma, valore in top_lmi:
    print(f"{bigramma}: {valore:.4f}")

# Numero di elementi comuni tra le due liste (top-10 MI e top-10 LMI)
bigrammi_mi = set(bigramma for bigramma, valore in top_mi)
bigrammi_lmi = set(bigramma for bigramma, valore in top_lmi)
comuni = bigrammi_mi & bigrammi_lmi
print(f"\nNumero di elementi comuni tra le due liste (MI e LMI): {len(comuni)}")

=== Top-10 bigrammi Agg+Sost per MI ===


('hearty', 'consent'): 14.0970
('quick', 'parts'): 14.0970
('emphatic', 'exclamation'): 14.0970
('noble', 'mien'): 14.0970
('accidental', 'recommendation'): 14.0970
('firmest', 'reliance'): 14.0970
('small', 'market'): 14.0970
('uniform', 'cheerfulness'): 14.0970
('similar', 'beforehand'): 14.0970
('distant', 'relation'): 14.0970

=== Top-10 bigrammi Agg+Sost per LMI ===
('young', 'man'): 44.0524
('smallest', 'objection'): 33.2909
('young', 'ladies'): 29.3683
('large', 'fortune'): 28.6579
('young', 'lady'): 27.0284
('olive', 'branch'): 25.0240
('humble', 'abode'): 25.0240
('same', 'time'): 24.6675
('great', 'deal'): 24.2016
('own', 'children'): 24.1577

Numero di elementi comuni tra le due liste (MI e LMI): 0


## 5. Analisi delle frasi (lunghezza 5-15 token, almeno metà non hapax)

In [10]:
# Frequenza di ogni token nel corpus (in minuscolo), usata per distinguere hapax e non hapax

freq_corpus = FreqDist(tokens_lower)

# Per ogni frase del corpus si ricavano i suoi token (in minuscolo, senza punteggiatura)
# Si selezionano poi le frasi che rispettano due condizioni:
#   - lunghezza compresa tra 5 e 15 token
#   - almeno la metà dei token NON è hapax 

frasi_idonee = []
for frase in frasi:
    token_frase = [t.lower() for t in word_tokenize(frase) if any(c.isalnum() for c in t)]

    if 5 <= len(token_frase) <= 15:
        non_hapax = [t for t in token_frase if freq_corpus[t] >= 2]
        if len(non_hapax) >= len(token_frase) / 2:
            frasi_idonee.append(token_frase)

print(f"Numero di frasi idonee: {len(frasi_idonee)}")
print("\nEsempio di frasi idonee:")
for frase in frasi_idonee[:3]:
    print(frase)

Numero di frasi idonee: 209

Esempio di frasi idonee:
['do', 'not', 'you', 'want', 'to', 'know', 'who', 'has', 'taken', 'it', 'cried', 'his', 'wife', 'impatiently']
['a', 'single', 'man', 'of', 'large', 'fortune', 'four', 'or', 'five', 'thousand', 'a', 'year']
['what', 'a', 'fine', 'thing', 'for', 'our', 'girls', 'how', 'so']


In [11]:
# Per ogni frase idonea si calcola la media della distribuzione di frequenza dei token:
# somma delle frequenze nel corpus dei token della frase, diviso il numero di token

def media_frequenze(frase, freq_corpus):
    somma_frequenze = sum(freq_corpus[t] for t in frase)
    return somma_frequenze / len(frase)

# Si associa a ogni frase la sua media
medie = [(frase, media_frequenze(frase, freq_corpus)) for frase in frasi_idonee]

# 5.1 - Frase con media più alta
frase_max = max(medie, key=lambda x: x[1])
print("=== Frase con media delle frequenze più ALTA ===")
print(f"Media: {frase_max[1]:.2f}")
print(f"Frase: {' '.join(frase_max[0])}")

# 5.2 - Frase con media più bassa
frase_min = min(medie, key=lambda x: x[1])
print("\n=== Frase con media delle frequenze più BASSA ===")
print(f"Media: {frase_min[1]:.2f}")
print(f"Frase: {' '.join(frase_min[0])}")

=== Frase con media delle frequenze più ALTA ===
Media: 242.20
Frase: a gentleman and a stranger

=== Frase con media delle frequenze più BASSA ===
Media: 32.00
Frase: arguments are too much like disputes


In [12]:
# Modello di Markov di ordine 3: la probabilità di una frase è data dalla
# probabilità del primo trigramma, moltiplicata per le probabilità condizionate
# P(parola | 3 parole precedenti) di ogni token successivo
# P(w4 | w1 w2 w3) = freq(w1 w2 w3 w4) / freq(w1 w2 w3)

# Distribuzioni di frequenza dei 3-grammi e 4-grammi sull'intero corpus (token in minuscolo)
freq_trigrammi = FreqDist(ngrams(tokens_lower, 3))
freq_quadrigrammi = FreqDist(ngrams(tokens_lower, 4))
N = len(tokens_lower)

# Funzione che calcola la probabilità di una frase secondo il modello di ordine 3
def probabilita_markov3(frase):
    trigrammi_frase = list(ngrams(frase, 3))
    quadrigrammi_frase = list(ngrams(frase, 4))

    # Se la frase è troppo corta per avere almeno un trigramma, non è valutabile
    if len(trigrammi_frase) == 0:
        return 0.0

    # Inizializzazione con la probabilità del primo trigramma: freq(primo trigramma) / N
    prob = freq_trigrammi[trigrammi_frase[0]] / N

    # Moltiplicazione per le probabilità condizionate di ogni token successivo
    for q in quadrigrammi_frase:
        trigramma_precedente = q[:3]
        freq_tri = freq_trigrammi[trigramma_precedente]
        if freq_tri == 0:
            prob_cond = 0.0
        else:
            prob_cond = freq_quadrigrammi[q] / freq_tri
        prob = prob * prob_cond
    return prob

# Calcolo della probabilità per ogni frase idonea e selezione della più probabile
prob_frasi = [(frase, probabilita_markov3(frase)) for frase in frasi_idonee]
frase_piu_probabile = max(prob_frasi, key=lambda x: x[1])

print("=== Frase con probabilità più alta secondo il modello di Markov di ordine 3 ===")
print(f"Probabilità: {frase_piu_probabile[1]}")
print(f"Frase: {' '.join(frase_piu_probabile[0])}")

=== Frase con probabilità più alta secondo il modello di Markov di ordine 3 ===
Probabilità: 0.00011413570735604634
Frase: i wish you had been there


## 6. Percentuale di stopwords

In [13]:
# Percentuale di stopwords nel corpus rispetto al totale dei token
# La lista di stopwords è quella fornita da NLTK per l'inglese

stopwords_eng = set(stopwords.words("english"))

# Conteggio dei token che sono stopwords (confronto in minuscolo)
num_stopwords = sum(1 for t in tokens_lower if t in stopwords_eng)

percentuale_stopwords = num_stopwords / len(tokens_lower) * 100

print(f"Numero di token: {len(tokens_lower)}")
print(f"Numero di stopwords: {num_stopwords}")
print(f"Percentuale di stopwords: {percentuale_stopwords:.2f}%")

Numero di token: 17523
Numero di stopwords: 9515
Percentuale di stopwords: 54.30%


## 7. Frequenza dei pronomi personali

In [14]:
# Frequenza d'uso dei pronomi personali nel corpus
# I pronomi personali sono identificati dal tag PoS "PRP"

# 7.1 - Numero di pronomi personali sul totale dei token
pronomi_personali = [token for token, tag in tokens_pos if tag == "PRP"]
num_pronomi = len(pronomi_personali)
percentuale_pronomi = num_pronomi / len(tokens_pos) * 100

# 7.2 - Numero medio di pronomi personali per frase
media_pronomi_frase = num_pronomi / len(frasi)

print(f"Numero di pronomi personali: {num_pronomi}")
print(f"Percentuale sul totale dei token: {percentuale_pronomi:.2f}%")
print(f"Numero medio di pronomi personali per frase: {media_pronomi_frase:.2f}")

Numero di pronomi personali: 1673
Percentuale sul totale dei token: 9.55%
Numero medio di pronomi personali per frase: 2.67


## 8. Named Entity Recognition

In [15]:
# Estrazione delle Named Entity (NE) del corpus con ne_chunk
# ne_chunk genera un albero in cui le entità nominate sono raggruppate in nodi
# che hanno un attributo 'label' contenente il tipo di entità

albero_NE = ne_chunk(tokens_pos)

# Le entità vengono raccolte in un dizionario: la chiave è il tipo di entità,
# il valore è la lista delle entità di quel tipo trovate nel corpus
entita_per_classe = {}
for nodo in albero_NE:
    if hasattr(nodo, 'label'):
        tipo = nodo.label()
        entita = " ".join(token for token, tag in nodo.leaves())
        if tipo not in entita_per_classe:
            entita_per_classe[tipo] = []
        entita_per_classe[tipo].append(entita)

# Per ciascuna classe si stampano le 20 entità più frequenti, ordinate per frequenza decrescente
for tipo, lista_entita in entita_per_classe.items():
    print(f"\n=== {tipo} (top-20) ===")
    for entita, freq in FreqDist(lista_entita).most_common(20):
        print(f"{entita}: {freq}")


=== PERSON (top-20) ===
Elizabeth: 61
Mr. Bingley: 43
Mr. Darcy: 41
Jane: 39
Miss Bingley: 31
Mr. Bennet: 29
Darcy: 23
Bingley: 20
Bennet: 12
Miss Bennet: 11
Mr. Collins: 11
Lady Catherine: 9
Lizzy: 8
Lydia: 8
Miss Lucas: 8
Mary: 7
Netherfield: 6
Sir William: 6
Mr. Hurst: 6
Mr.: 5

=== GPE (top-20) ===
London: 6
Meryton: 6
Longbourn: 5
Netherfield: 5
Lydia: 4
England: 3
Long: 2
Derbyshire: 1
Darcy: 1
Irish: 1
Captain: 1
Clarke: 1
Italian: 1
Scotch: 1
Elizabeth: 1
Bourgh: 1
British: 1

=== ORGANIZATION (top-20) ===
Netherfield: 6
Meryton: 4
Colonel: 4
Elizabeth: 2
St. James: 2
Pemberley: 2
Longbourn: 2
Rosings: 2
least.: 1
Impossible: 1
Come Darcy: 1
Miss Bennet: 1
Bennets: 1
Miss Lucases: 1
Miss Bennets: 1
Miss Bingley: 1
Vanity: 1
Commerce: 1
Sir: 1
Meryton With: 1

=== FACILITY (top-20) ===
Netherfield House: 1


## 9. LMI delle Named Entity multi-token

In [16]:
# Per le Named Entity composte da almeno due token si calcola la LMI tra i loro token
# LMI(w1...wn) = freq(w1...wn) * log( p(w1...wn) / (p(w1)*p(w2)*...*p(wn)) )
# dove le probabilità sono stimate come frequenza nel corpus diviso il numero totale di token

# Frequenza dei singoli token e dimensione del corpus (token in minuscolo)
freq_token = FreqDist(tokens_lower)
N = len(tokens_lower)

# Raccolta delle Named Entity composte da almeno due token
# Ogni entità è rappresentata come lista dei suoi token (in minuscolo)
ne_multitoken = []
for nodo in albero_NE:
    if hasattr(nodo, 'label'):
        token_entita = [token.lower() for token, tag in nodo.leaves()]
        if len(token_entita) >= 2:
            ne_multitoken.append(token_entita)

# Frequenza di ogni entità multi-token, calcolata sul numero di volte in cui
# la sequenza è stata riconosciuta come Named Entity nel corpus
freq_ne = FreqDist(tuple(ne) for ne in ne_multitoken)

# Calcolo della LMI per ogni entità multi-token distinta
risultati_lmi = {}
for entita in set(tuple(ne) for ne in ne_multitoken):
    freq_congiunta = freq_ne[entita]
    p_congiunta = freq_congiunta / N

    # Prodotto delle probabilità dei singoli token
    p_singoli = 1.0
    for token in entita:
        p_singoli = p_singoli * (freq_token[token] / N)

    lmi_valore = freq_congiunta * math.log2(p_congiunta / p_singoli)
    risultati_lmi[entita] = lmi_valore

# Stampa delle entità ordinate per LMI decrescente
print("=== LMI delle Named Entity composte da almeno 2 token ===")
for entita, valore in sorted(risultati_lmi.items(), key=lambda x: x[1], reverse=True):
    print(f"{' '.join(entita)}: {valore:.4f}")

=== LMI delle Named Entity composte da almeno 2 token ===
mr. darcy: 249.0806
mr. bingley: 246.1296
miss bingley: 200.8890
mr. bennet: 155.4038
lady catherine: 77.4460
mr. collins: 76.1974
sir william: 63.3680
miss bennet: 60.7900
miss lucas: 53.2737
sir lewis de bourgh: 33.8986
mr. hurst: 32.5472
right honourable lady catherine: 31.0441
colonel forster: 29.6237
st. james: 26.1939
lady lucas: 20.0373
miss bennet miss bingley: 16.8556
charlotte mr darcy: 16.6625
sir william lucas: 16.0942
dear lizzy oh: 15.7021
charlotte lucas: 15.0542
william lucas: 14.7792
miss bingley charles: 14.3747
mr. robinson: 13.8541
miss eliza bennet: 12.5778
miss eliza: 11.8143
rosings park: 11.5120
mr. philips: 11.2102
lizzy mr. bingley: 11.0727
captain carter: 10.9270
lucas lodge: 9.8490
maria lucas: 9.8490
well jane: 9.1001
poor eliza: 9.0970
netherfield park: 8.7750
would mr. darcy: 8.6446
miss grantley: 7.9071
miss watson: 7.9071
mr. morris: 6.9270
mr. denny: 6.9270
mr. jones: 6.9270
miss king: 6.9071
mi